In [1]:
import os
import glob
import numpy as np

In [ ]:
# Parámetros de configuración
current_dataset_dir = 'data'      # Carpeta del dataset original
desired_total = 115000            # Total deseado después de la ampliación
num_points = 5000                 # Cada espectro tiene 5000 puntos

# Rango de redshift para seleccionar nuevos espectros
z_min = 3.0
z_max = 3.6

# Cargar los redshifts existentes del dataset actual
current_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_100k_redshift_mmap.dat')
current_total = 100000  # Número actual de espectros
current_redshifts = np.memmap(current_redshift_path, dtype='float32', mode='r', shape=(current_total,))
existing_redshifts = set(current_redshifts.tolist())

# Cargar el nuevo dataset (3M espectros) desde archivos .dat
# Asumimos que los archivos nuevos se han guardado con np.memmap y tienen las siguientes rutas:
new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_2d.dat')
new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_2d.dat')
new_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_complete_redshift_mmap.dat')

new_total_available = 3372890  # Total de espectros en el nuevo dataset

# Cargar los memmaps del nuevo dataset en modo lectura
new_flux = np.memmap(new_flux_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_wavelength = np.memmap(new_wavelength_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_redshifts = np.memmap(new_redshift_path, dtype='float32', mode='r', shape=(new_total_available,))

new_flux_list = []
new_wavelength_list = []
new_redshift_list = []

print("Filtrando nuevos espectros del dataset de 3M...")
# Recorrer el nuevo dataset
for i in range(new_total_available):
    r = new_redshifts[i]
    # Filtrar por rango de redshift
    if r < z_min or r > z_max:
        continue
    # Evitar duplicados: se comprueba que el redshift no exista ya en el dataset original
    if r in existing_redshifts:
        continue
    # Si cumple ambas condiciones, se añade la información
    new_flux_list.append(new_flux[i, :].copy())         # .copy() para obtener un array independiente
    new_wavelength_list.append(new_wavelength[i, :].copy())
    new_redshift_list.append(r)
    existing_redshifts.add(r)  # Agregar para evitar duplicados posteriores
    
    # Si se alcanza el total deseado, se finaliza el filtrado
    if current_total + len(new_redshift_list) >= desired_total:
        break

print(f"Se han encontrado {len(new_redshift_list)} nuevos espectros en el rango de redshift [{z_min}, {z_max}].")

# Crear el nuevo dataset ampliado
new_total = current_total + len(new_redshift_list)
print(f"Dataset ampliado: {new_total} espectros.")

# Rutas para los nuevos archivos memmap actualizados
new_flux_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap2.dat')
new_wavelength_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_wavelength_mmap2.dat')
new_redshift_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_redshift_mmap2.dat')

# Preasignar los memmaps para el dataset ampliado
flux_mmap_new = np.memmap(new_flux_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
wavelength_mmap_new = np.memmap(new_wavelength_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
redshift_mmap_new = np.memmap(new_redshift_mmap_path, dtype='float32', mode='w+', shape=(new_total,))

# Copiar los datos antiguos del dataset actual
current_flux_path = os.path.join(current_dataset_dir, 'spectra_data_100k_flux_2d.dat')
current_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_100k_wavelength_2d.dat')

flux_mmap_old = np.memmap(current_flux_path, dtype='float32', mode='r', shape=(current_total, num_points))
wavelength_mmap_old = np.memmap(current_wavelength_path, dtype='float32', mode='r', shape=(current_total,))

# Copiar los datos del dataset original en los nuevos memmaps
flux_mmap_new[:current_total, :] = flux_mmap_old[:]
wavelength_mmap_new[:current_total, :] = wavelength_mmap_old[:]
redshift_mmap_new[:current_total] = current_redshifts[:]

# Añadir los nuevos espectros filtrados
for i, (flux_array, wave_array, r) in enumerate(zip(new_flux_list, new_wavelength_list, new_redshift_list)):
    idx = current_total + i
    flux_mmap_new[idx, :] = flux_array
    wavelength_mmap_new[idx, :] = wave_array
    redshift_mmap_new[idx] = r

# Asegurarse de que los cambios se guarden en disco
flux_mmap_new.flush()
wavelength_mmap_new.flush()
redshift_mmap_new.flush()

print("Se ha actualizado el dataset ampliado y se han guardado los nuevos archivos memmap.")

Filtrando nuevos espectros del dataset de 3M...
Se han encontrado 15000 nuevos espectros en el rango de redshift [3.0, 3.6].
Dataset ampliado: 115000 espectros.


ValueError: could not broadcast input array from shape (100000,) into shape (100000,5000)

In [ ]:
# Parámetros de configuración
current_dataset_dir = 'data'      # Carpeta del dataset original
desired_total = 500000            # Total deseado después de la ampliación
num_points = 5000                 # Cada espectro tiene 5000 puntos

# Rango de redshift para seleccionar nuevos espectros
z_min = 0.0
z_max = 8.6

# Cargar los redshifts existentes del dataset actual
current_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_500k_redshift_mmap2.dat')
current_total = 115000  # Número actual de espectros
current_redshifts = np.memmap(current_redshift_path, dtype='float32', mode='r', shape=(current_total,))
existing_redshifts = set(current_redshifts.tolist())

# Cargar el nuevo dataset (3M espectros) desde archivos .dat
# Asumimos que los archivos nuevos se han guardado con np.memmap y tienen las siguientes rutas:
new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_2d.dat')
new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_2d.dat')
new_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_complete_redshift_mmap.dat')

new_total_available = 3372890  # Total de espectros en el nuevo dataset

# Cargar los memmaps del nuevo dataset en modo lectura
new_flux = np.memmap(new_flux_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_wavelength = np.memmap(new_wavelength_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_redshifts = np.memmap(new_redshift_path, dtype='float32', mode='r', shape=(new_total_available,))

new_flux_list = []
new_wavelength_list = []
new_redshift_list = []

print("Filtrando nuevos espectros del dataset de 3M...")
# Recorrer el nuevo dataset
for i in range(new_total_available):
    r = new_redshifts[i]
    # Filtrar por rango de redshift
    if r < z_min or r > z_max:
        continue
    # Evitar duplicados: se comprueba que el redshift no exista ya en el dataset original
    if r in existing_redshifts:
        continue
    # Si cumple ambas condiciones, se añade la información
    new_flux_list.append(new_flux[i, :].copy())         # .copy() para obtener un array independiente
    new_wavelength_list.append(new_wavelength[i, :].copy())
    new_redshift_list.append(r)
    existing_redshifts.add(r)  # Agregar para evitar duplicados posteriores
    
    # Si se alcanza el total deseado, se finaliza el filtrado
    if current_total + len(new_redshift_list) >= desired_total:
        break

print(f"Se han encontrado {len(new_redshift_list)} nuevos espectros en el rango de redshift [{z_min}, {z_max}].")

# Crear el nuevo dataset ampliado
new_total = current_total + len(new_redshift_list)
print(f"Dataset ampliado: {new_total} espectros.")

# Rutas para los nuevos archivos memmap actualizados
new_flux_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap.dat')
new_wavelength_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_wavelength_mmap.dat')
new_redshift_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_redshift_mmap.dat')

# Preasignar los memmaps para el dataset ampliado
flux_mmap_new = np.memmap(new_flux_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
wavelength_mmap_new = np.memmap(new_wavelength_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
redshift_mmap_new = np.memmap(new_redshift_mmap_path, dtype='float32', mode='w+', shape=(new_total,))

# Copiar los datos antiguos del dataset actual
current_flux_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap2.dat')
current_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_500k_wavelength_mmap2.dat')

flux_mmap_old = np.memmap(current_flux_path, dtype='float32', mode='r', shape=(current_total, num_points))
wavelength_mmap_old = np.memmap(current_wavelength_path, dtype='float32', mode='r', shape=(current_total,))

# Copiar los datos del dataset original en los nuevos memmaps
flux_mmap_new[:current_total, :] = flux_mmap_old[:]
wavelength_mmap_new[:current_total, :] = wavelength_mmap_old[:]
redshift_mmap_new[:current_total] = current_redshifts[:]

# Añadir los nuevos espectros filtrados
for i, (flux_array, wave_array, r) in enumerate(zip(new_flux_list, new_wavelength_list, new_redshift_list)):
    idx = current_total + i
    flux_mmap_new[idx, :] = flux_array
    wavelength_mmap_new[idx, :] = wave_array
    redshift_mmap_new[idx] = r

# Asegurarse de que los cambios se guarden en disco
flux_mmap_new.flush()
wavelength_mmap_new.flush()
redshift_mmap_new.flush()

print("Se ha actualizado el dataset ampliado y se han guardado los nuevos archivos memmap.")

In [5]:
import os
import numpy as np

# Ruta al archivo memmap de longitudes de onda
current_dataset_dir = 'data'
wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_2d.dat')

# Cargar el memmap sin especificar la forma (asumiendo que fue guardado con np.memmap sin shape)
try:
    wavelength_memmap = np.memmap(wavelength_path, dtype='float32', mode='r')
    print("Shape raw del memmap:", wavelength_memmap.shape)
except Exception as e:
    print("Error al cargar el memmap:", e)

# Obtener el tamaño del archivo en bytes
file_size = os.path.getsize(wavelength_path)
print("Tamaño del archivo (bytes):", file_size)

# Supongamos que sabemos que current_total es 100000 y cada espectro debe tener 5000 puntos
current_total = 3372890
num_points = 5000

# # Analizamos la dimensión del array:
# if wavelength_memmap.ndim == 1:
#     print("El array es 1D con longitud:", wavelength_memmap.shape[0])
#     if wavelength_memmap.shape[0] == num_points:
#         print("El array contiene la grilla de longitudes de onda común a todos los espectros.")
#         # Si es así, debes replicarlo para que tenga forma (current_total, num_points):
#         wavelength_corrected = np.tile(wavelength_memmap, (current_total, 1))
#         print("Nuevo shape tras replicar:", wavelength_corrected.shape)
#     else:
#         print("La longitud del array 1D no coincide con num_points. Revisa cómo se guardó el archivo.")
# else:
#     print("El array es de {0}D con shape: {1}".format(wavelength_memmap.ndim, wavelength_memmap.shape))

Shape raw del memmap: (16864450000,)
Tamaño del archivo (bytes): 67457800000


In [9]:
import os
import numpy as np

# Parámetros de configuración
current_dataset_dir = 'data'
current_total = 3372890  # Número de espectros
num_points = 5000       # Número de puntos por espectro

# Rutas de los archivos originales (1D)
flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_mmap.dat')
wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_mmap.dat')

# Cargar los memmaps en modo lectura (sin especificar shape, se obtiene la forma raw)
flux_memmap_raw = np.memmap(flux_path, dtype='float32', mode='r')
wavelength_memmap_raw = np.memmap(wavelength_path, dtype='float32', mode='r')

print("Flux raw shape:", flux_memmap_raw.shape)
print("Wavelength raw shape:", wavelength_memmap_raw.shape)

# Convertir a arrays 2D con shape (100000, 5000)
flux_2d = flux_memmap_raw.reshape((current_total, num_points))
wavelength_2d = wavelength_memmap_raw.reshape((current_total, num_points))

print("Flux 2D shape:", flux_2d.shape)
print("Wavelength 2D shape:", wavelength_2d.shape)

# Opcional: guardar los nuevos arrays 2D en nuevos archivos memmap
new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_2d.dat')
new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_2d.dat')

flux_memmap_2d = np.memmap(new_flux_path, dtype='float32', mode='w+', shape=(current_total, num_points))
wavelength_memmap_2d = np.memmap(new_wavelength_path, dtype='float32', mode='w+', shape=(current_total, num_points))

flux_memmap_2d[:] = flux_2d[:]
wavelength_memmap_2d[:] = wavelength_2d[:]

flux_memmap_2d.flush()
wavelength_memmap_2d.flush()

print("Nuevos archivos memmap 2D creados:")
print(f"  Flux: {new_flux_path}")
print(f"  Wavelength: {new_wavelength_path}")

Flux raw shape: (16864450000,)
Wavelength raw shape: (16864450000,)
Flux 2D shape: (3372890, 5000)
Wavelength 2D shape: (3372890, 5000)
Nuevos archivos memmap 2D creados:
  Flux: data\spectra_data_complete_flux_2d.dat
  Wavelength: data\spectra_data_complete_wavelength_2d.dat
